# سبق 13 - ایجنٹ میموری


## ترتیب

یہ نوٹ بک دکھاتی ہے کہ **Microsoft Agent Framework** (MAF) استعمال کرتے ہوئے **مستقل یادداشت** کے ساتھ سفر کی بکنگ ایجنٹ کیسے بنائی جائے۔

آپ سیکھیں گے کہ ایجنٹ کی مختلف قسم کی یادداشتیں — کام کرنے والی، قلیل مدتی، اور طویل مدتی — ایجنٹ کے بات چیت کے دوران معلومات کو کیسے محفوظ اور استعمال کرنے کو متاثر کرتی ہیں۔

**ضروریات:**
- ایک Microsoft Foundry پراجیکٹ جس کا چیٹ ماڈل تعینات ہو (مثلاً `gpt-4.1-mini`)۔
- Azure CLI میں لاگ ان — اپنے ٹرمینل میں `az login` چلائیں۔
- `AZURE_AI_PROJECT_ENDPOINT` — آپ کے Microsoft Foundry پراجیکٹ کا اینڈ پوائنٹ۔
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — آپ کے تعینات کردہ ماڈل کا نام۔


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ایجنٹ میموری کی اقسام

AI ایجنٹس مختلف اقسام کی میموری استعمال کر سکتے ہیں، ہر ایک مختلف مقصد کے لیے خدمت انجام دیتا ہے:

### ورکنگ میموری
گفتگو کا سلسلہ خود — ایک ہی سیشن میں تبادلہ کی گئی پیغامات۔ ایجنٹ اسی سلسلے میں پہلے کے پیغامات کا حوالہ دے سکتا ہے تاکہ تسلسل برقرار رہے۔ MAF میں یہ **`agent.create_session()`** کے ذریعے بنایا جاتا ہے، جو `AgentSession` واپس کرتا ہے۔

### قلیل مدتی میموری
ایسی معلومات جو کام یا سیشن کے دوران برقرار رہتی ہیں لیکن مستقل طور پر محفوظ نہیں ہوتیں۔ مثال کے طور پر، ایجنٹ ایک کثیر دورانیے کی منصوبہ بندی کی گفتگو کے دوران حقائق جمع کر سکتا ہے اور انہیں حتمی منصوبہ تیار کرنے کے لیے استعمال کر سکتا ہے۔

### طویل مدتی میموری
ترجیحات اور حقائق جو **سیشنز کے درمیان** قائم رہتے ہیں۔ واپس آنے والے صارف کو اپنی خوراک کی پابندیاں یا سفر کا انداز بار بار دہرانا نہیں پڑنا چاہیے۔ طویل مدتی میموری عام طور پر بیرونی ذخیرہ — ڈیٹا بیس، فائل، یا ویکٹر انڈیکس — کے ذریعے معاون ہوتی ہے اور ٹولز کے ذریعے ایجنٹ کو فراہم کی جاتی ہے۔


## سیشنز کے ساتھ کام کرنے والی میموری

میموری کی سب سے آسان صورت گفتگو کا سیشن ہے۔ جب آپ ایک ہی سیشن (جو `agent.create_session()` کے ذریعے بنایا گیا ہو) کو لگاتار `agent.run()` کالز میں دیتے ہیں، تو ایجنٹ اُس گفتگو کی مکمل تاریخ دیکھ سکتا ہے اور پہلے کے تفصیلات کو یاد رکھ سکتا ہے۔

آئیے ایک سفری ایجنٹ بنائیں اور کام کرنے والی میموری کا مظاہرہ کریں۔


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ایجنٹ نے بجٹ کو صحیح طور پر یاد رکھا کیونکہ دونوں پیغامات ایک ہی سیشن کا حصہ ہیں۔ یہ **ورکنگ میموری** ہے — یہ صرف سیشن کی مدت کے لیے موجود رہتی ہے۔

### نئے تھریڈ کے ساتھ کیا ہوتا ہے؟

اگر ہم ایک **نیا** سیشن بنائیں تو ایجنٹ کو پچھلی گفتگو کی کوئی یادداشت نہیں ہوتی:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## طویل مدتی یادداشت کا نمونہ

صارف کی ترجیحات کو **سیشنز کے درمیان** یاد رکھنے کے لیے، ہمیں ایک مستقل اسٹور کی ضرورت ہے جو گفتگو کے دھاگے کے باہر موجود ہو۔ ایجنٹ اس اسٹور تک رسائی **ٹولز** کے ذریعے حاصل کرتا ہے — یعنی وہ فنکشنز جنہیں وہ معلومات کو محفوظ کرنے اور دوبارہ حاصل کرنے کے لیے کال کر سکتا ہے۔

نیچے ہم ایک سادہ میموری میں ترجیحاتی اسٹور نافذ کرتے ہیں (پروڈکشن میں آپ اس کے لیے کسی ڈیٹا بیس یا ویکٹر انڈیکس کا استعمال کریں گے) اور اسے ٹولز کے طور پر ظاہر کرتے ہیں جنہیں ایجنٹ استعمال کر سکتا ہے۔

### ساخت
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### منظر نامہ 1 — پہلی بار استعمال کنندہ سالگرہ کی سفر بک کرتا ہے

سارہ پہلی بار آتی ہے۔ ایجنٹ کو چاہیے کہ وہ اس کی ترجیحات ٹولز کے ذریعے محفوظ کرے اور انہیں ہوٹلوں کی سفارش کے لیے استعمال کرے۔


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### منظر نامہ 2 — سارہ ہفتوں بعد واپس آتی ہے

سارہ ایک **نئی تھریڈ** شروع کرتی ہے (نئی سیشن کی نقل کرتے ہوئے)۔ ورکنگ میموری خالی ہے، لیکن طویل مدتی ترجیح اسٹور میں اب بھی اس کی معلومات موجود ہیں۔ ایجنٹ کو یہ معلومات بازیافت کرنی چاہیے اور سفارشات کو ذاتی نوعیت دینے کے لیے استعمال کرنا چاہیے۔


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## خلاصہ

اس سبق میں آپ نے ایجنٹ کی تین اقسام کی یادداشت سیکھی اور انہیں مائیکروسافٹ ایجنٹ فریم ورک کے ساتھ کیسے نافذ کریں:

| یادداشت کی قسم | MAF میکانزم | عمر |
|---|---|---|
| **کام کرنے والی** | `agent.create_session()` | ایک گفتگو کے دوران |
| **مختصر مدت** | تھریڈ میں جمع کیا گیا کانٹیکسٹ | ایک کام / سیشن کے دوران |
| **طویل مدت** | `@tool` فنکشنز کے ذریعے رسائی والا خارجی اسٹور | سیشنز کے دوران |

### اہم نکات
1. **`agent.create_session()`** کام کرنے والی یادداشت فراہم کرتا ہے — ایجنٹ ایک سیشن کے اندر پوری گفتگو کا تاریخ دیکھتا ہے۔
2. **نئے سیشنز کانٹیکسٹ کھو دیتے ہیں** — طویل مدتی یادداشت کے بغیر ایجنٹ پرانی بات چیت یاد نہیں کر سکتا۔
3. **`@tool` فنکشنز** خلا کو پر کرتے ہیں — یہ ایجنٹ کو مستقل اسٹور سے معلومات محفوظ اور بازیافت کرنے دیتے ہیں۔
4. **ذاتی موافقت وقت کے ساتھ بہتر ہوتی ہے** — جتنی زیادہ ترجیحات محفوظ ہوں گی، ایجنٹ کی سفارشات اتنی ہی بہتر ہوں گی۔

### حقیقی دنیا کی درخواستیں
- **کسٹمر سروس**: کسٹمر کی تاریخ اور ترجیحات یاد رکھیں
- **ذاتی معاون**: دنوں یا ہفتوں کے دوران کانٹیکسٹ برقرار رکھیں
- **صحت کی دیکھ بھال**: مریض کی معلومات اور ترجیحات کا پتہ رکھیں
- **ای کامرس**: تاریخ کی بنیاد پر ذاتی نوعیت کی خریداری

### اگلے اقدامات
- یادداشت کے ان-میموری ڈکشنری کو ڈیٹا بیس یا ویکٹر اسٹور (مثلاً Azure AI سرچ) سے تبدیل کریں
- وقت کے حساس معلومات کے لیے یادداشت کی معیاد مقرر کریں
- مشترکہ یادداشت کے ساتھ ملٹی ایجنٹ سسٹمز تیار کریں
- نالج گراف پر مبنی یادداشت کے لیے Cognee نوٹ بک دریافت کریں


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
